# Example 0.2 / 1: See the Feasible Region of a Two-Variable Model

Companion notebook for **0.2 Optimization Basics** in the AMPL lab Documentation, section "See the Feasible Region of a Two-Variable Model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## See the Feasible Region of a Two-Variable Model

With two decisions, the feasible region can be drawn in the plane. A lemonade stand sells
lemonade $x$ at 1.50 a glass and iced tea $y$ at 1.00 a glass. A glass of lemonade uses one lemon
and two cups of sugar; a glass of iced tea uses one tea bag and one cup of sugar. Each day there
are 10 lemons, 8 tea bags and 19 cups of sugar:

$$
\begin{aligned}
\text{maximize}\quad & 1.5x + y \
\text{subject to}\quad & x \le 10, \quad y \le 8, \quad 2x + y \le 19, \quad x, y \ge 0.
\end{aligned}
$$

Each constraint keeps one side of a line, and the feasible region is the polygon where all of
them hold. A linear objective is best at a corner of that polygon, so the corners can be checked
by hand:

| Corner $(x, y)$ | Where it lies | Revenue $1.5x + y$ |
|---|---|---|
| $(0, 0)$ | origin | 0 |
| $(9.5, 0)$ | sugar line meets $y = 0$ | 14.25 |
| $(5.5, 8)$ | sugar line meets the tea-bag line | 16.25 |
| $(0, 8)$ | tea-bag line meets $x = 0$ | 8 |

The lemon limit $x \le 10$ never binds, because sugar allows at most 9.5 glasses of lemonade.
The best corner, $(5.5, 8)$, sells half a glass. If only whole glasses can be sold, the
variables must be integers, and the integer optimum is not the LP optimum rounded down:
$(5, 8)$ earns 15.50, while $(6, 7)$ uses $2 \cdot 6 + 7 = 19$ cups of sugar and earns 16.

The model below declares `Make` as `integer`. The AMPL option `relax_integrality` set to 1 tells
AMPL to ignore integrality and solve the LP; set to 0 it solves the integer model. In `run.py`,
`ampl` has already read `model.mod` (Running AMPL from Python (AMPL lab documentation, section 0.5)):

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # drinks
set INGR;                         # dry ingredients
param price {PROD} > 0;           # price per glass
param limit {INGR} >= 0;          # units available per day
param usage {INGR, PROD} >= 0;    # units used per glass
var Make {PROD} integer >= 0;     # glasses made
maximize Revenue: sum {p in PROD} price[p] * Make[p];
subject to Ingredients {i in INGR}:  # stay within each ingredient
    sum {p in PROD} usage[i,p] * Make[p] <= limit[i];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["lemonade", "iced_tea"]
ampl.set["INGR"] = ["lemon", "tea_bag", "sugar"]
ampl.param["price"] = {"lemonade": 1.5, "iced_tea": 1}
ampl.param["limit"] = {"lemon": 10, "tea_bag": 8, "sugar": 19}
ampl.param["usage"] = {("lemon", "lemonade"): 1, ("lemon", "iced_tea"): 0,
                       ("tea_bag", "lemonade"): 0, ("tea_bag", "iced_tea"): 1,
                       ("sugar", "lemonade"): 2, ("sugar", "iced_tea"): 1}

In [5]:
# run.py
for relax in [1, 0]:
    ampl.option["relax_integrality"] = relax
    ampl.solve(solver="highs", verbose=False)
    print("relax_integrality", relax, ampl.var["Make"].to_dict(), ampl.obj["Revenue"].value())

relax_integrality 1 {'iced_tea': 8, 'lemonade': 5.5} 16.25
relax_integrality 0 {'iced_tea': 7, 'lemonade': 6} 16.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := lemonade iced_tea;
set INGR := lemon tea_bag sugar;
param price := lemonade 1.5  iced_tea 1;
param limit := lemon 10  tea_bag 8  sugar 19;
param usage:  lemonade iced_tea :=
  lemon          1        0
  tea_bag        0        1
  sugar          2        1 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
option relax_integrality 1;
solve;
display Make, Revenue;
option relax_integrality 0;
solve;
display Make, Revenue;

Make [*] :=
iced_tea  8
lemonade  5.5
;

Revenue = 16.25

Make [*] :=
iced_tea  7
lemonade  6
;

Revenue = 16

